In [1]:
import matplotlib.pyplot as plt
import numpy as np

# **Metodo agli Elementi Finiti (stazionario) - Parte 2**

## Condizioni di Neumann

Per problemi mono-dimensionali, definiti su un certo intervallo $[a,b]\subset\mathbb{R}$, una condizione al bordo della forma</br></br>
$$u'(b)=\gamma,$$

o, in alternativa, $u'(a)=\gamma$, è detta condizione di Neumann. Nei metodi agli elementi finiti, questo tipo di condizione viene tipicamente gestita includendo esplicitamente un termine di bordo nella formulazione variazionale. Ad esempio, si consideri il seguente problema con condizioni miste Dirichlet-Neumann,

$$\begin{cases}
-u'' = f, & \text{in}\;(a,b)\\\\
u(a)=\alpha,\;\;
u'(b)=\gamma.
\end{cases}$$

Posto $V_{\text{test}}:=\{v\in H^{1}(a,b)\;|\;v(a)=0\}$, la sua formulazione debole è

$$\int_a^b-u''vdx = \int_a^b fvdx, \quad\quad \forall v\in V_{\text{test}}\quad\rightsquigarrow\quad\int_a^bu'v'dx - \left[u'v\right]\Big |^{b}_{a}= \int_a^b fvdx, \quad\quad \forall v\in V_{\text{test}}$$

$$\rightsquigarrow\int_a^bu'v'dx = \int_a^b fvdx + \left[u'v\right]\Big |^{b}_{a},\quad\quad \forall v\in V_{\text{test}}$$

$$\rightsquigarrow\int_a^bu'v'dx = \int_a^b fvdx + \gamma v(b),\quad\quad \forall v\in V_{\text{test}}.$$

Il termine $\gamma v(b)$ produce quindi una modifica in corrispondenza dell'ultimo nodo della mesh, che va gestita opportunamente.

<mark>**Esercizio 1.1**</mark></br>

Si consideri il seguente problema differenziale

$$\begin{cases}
-u''=30x, & x\in(0,1)\\
u(0)=0,\\u'(1)=3.
\end{cases}$$

Risolvere il problema implementando il metodo agli elementi finiti (grado polinomiale $r=1$, passo della mesh $h=0.1$). Confrontare graficamente la soluzione ottenuta con la soluzione esatta, $u(x)=18x-5x^3.$

In [ ]:
from fem_utils import Grid, fun2dof, create_restriction, diffusion, mass

# estremi del dominio
a = 0
b = 1
# dimensione elementi
h = 0.1
# bc di Neumann
gamma = 3

# generazione griglia
...

# assemblaggio matrice di rigidezza
...

# assemblaggio termine noto con condizioni di Neumann
...

# Neumann al bordo destro
...

# gestione condizioni di Dirichlet
dirichlet_nodes = [...]
dirichlet_values = [...]

# creazione vettore dei valori al bordo, inizializzato a zero
bc_values = np.zeros(Nele + 1)
bc_values[dirichlet_nodes] = dirichlet_values

# costruzione matrice di restrizione R
keep_dof = np.ones(Nele + 1, dtype=bool)
keep_dof[dirichlet_nodes] = False

R = ...

# Restrizione di A e del termine noto
...

# risoluzione sistema lineare
...

# aggiunta condizione di Dirichlet
...

In [ ]:
# rappresentazione grafica della soluzione numerica e confronto con soluzione esatta

# soluzione esatta
uex = lambda x: 18 * x - 5 * (x**3)
xplot = np.linspace(a, b, 1000)

plt.figure(figsize=(5, 4))
plt.plot(grid.nodes, u, marker=".", label="Soluzione numerica")
plt.plot(xplot, uex(xplot), "--", label="Soluzione esatta")

plt.title("Confronto tra soluzione numerica e soluzione esatta")
plt.legend()
plt.show()

<mark>**Esercizio 1.2**</mark></br>

Ripetere l'Es. 1.1 invertendo le condizioni di Neumann e Dirichlet, cioè risolvendo

$$\begin{cases}
-u''=30x & x\in(0,1)\\
u'(0)=3\\u(1)=0,
\end{cases}$$

la cui soluzione esatta è $u(x)=3x-5x^3+2.$

In [ ]:
from fem_utils import Grid, fun2dof, create_restriction, diffusion, mass

# estremi del dominio
a = 0
b = 1
# dimensione elementi
h = 0.1
# numero elementi
Nele = int((b - a) / h)
# creazione griglia
...

# assemblaggio matrice di rigidezza
...

# assemblaggio termine noto con condizioni di Neumann
...

# gestione condizioni di Dirichlet
dirichlet_nodes = [...]
dirichlet_values = [...]

bc_values = np.zeros(Nele + 1)
bc_values[dirichlet_nodes] = dirichlet_values
keep_dof = np.ones(Nele + 1, dtype=bool)
keep_dof[dirichlet_nodes] = False

R = create_restriction(keep_dof)
print(R)

A_0 = ...
rhs_0 = ...

# risoluzione sistema lineare
u_0 = ...

# aggiunta condizione di Dirichlet
u = ...

In [ ]:
# rappresentazione grafica della soluzione numerica e confronto con soluzione esatta
# soluzione esatta
uex = lambda x: 3 * x - 5 * (x**3) + 2
xplot = np.linspace(a, b, 1000)


plt.figure(figsize=(4, 3))
plt.plot(grid.nodes, u, marker=".", label="Soluzione numerica")
plt.plot(xplot, uex(xplot), "--", label="Soluzione esatta")

plt.title("Confronto tra soluzione numerica e soluzione esatta")
plt.legend()
plt.show()

# Equazioni di diffusione-trasporto (esempio di trasporto dominante)

Si consideri il seguente problema differenziale, descrivente un fenomeno di diffusione-trasporto (stazionario)

$$
\begin{cases}
(-\alpha u'+\beta u)'= f, & x\in(0,1)\\
u(0)=0\\
u(1)=1.
\end{cases}
$$

dove $\alpha>0,\beta\neq0$ sono opportuni coefficienti. La corrispondente formulazione debole è
</br></br>
$$ \int_0^1 \alpha u'v'dx-\int_0^1\beta uv'dx=\int_0^1fvdx.$$

<mark>**Esercizio 2.1**</mark></br>

Sia $V_h$ lo spazio elementi finiti di grado $r=1$ e passo $h=0.1$. 

Costruire la mesh del problema e asseblare le matrici del problema associate alle forme bilineari e ai funzionali lineari, senza imporre alcuna condizione al bordo, cioè costruire le seguenti matrici e vettore

$$A_{\text{diff}}:=\int u'v'dx,\quad\quad
A_{\text{trasp}}:=-\int uv'dx,\quad\quad
F:=\int fvdx
$$

### Struttura delle matrici da assemblare

La matrice di diffusione e quella di reazione (detta anche matrice di massa), che usiamo per assemblare la forzante, le conosciamo già (si vedano Lab 10 e appunti di teoria). 

Per il trasporto, come visto a teoria, otteniamo invece la seguente matrice:
$$
\mathbf{A}_{\mathrm{trasp}}=
\begin{bmatrix}
0 & \tfrac12 & 0 & \cdots & 0\\
-\tfrac12 & 0 & \tfrac12 & \ddots & \vdots\\
0 & -\tfrac12 & 0 & \ddots & 0\\
\vdots & \ddots & \ddots & \ddots & \tfrac12\\
0 & \cdots & 0 & -\tfrac12 & 0
\end{bmatrix}.
$$

**Commento**: si osserva che, in presenza di coefficienti costanti, i termini di diffusione e trasporto possono essere riscritti come segue:

\begin{equation}
(-\alpha u'+\beta u)' = -\alpha u'' + \beta u'.
\end{equation}

In tal caso, passando alla formulazione debole, il termine di traporto può sia essere trattato integrando per parti, sia non integrando. In quest'ultimo caso, $a_{\text{trasp}}(u,v):=\int u'vdx,$ che però dà luogo alla stessa matrice $A_{\mathrm{trasp}}$ vista in precedenza.

In [ ]:
from fem_utils import Grid, fun2dof, create_restriction, diffusion, transport, mass

# estremi del dominio
a = 0
b = 1
# dimensione elementi
h = 0.1

# costruzione griglia
Nele = int((b - a) / h)

# creazione griglia
grid = Grid(a, b, Nele)
grid.compute_geometry()

# assemblaggio matrici
...

<mark>**Esercizio 2.2**</mark></br>

Si verifichi che le matrici di diffusione e di massa (reazione) risultano simmetriche, mentre la matrice di trasporto non è simmetrica.

In [ ]:
...

<mark>**Esercizio 2.3**</mark></br>

Sfruttando le matrici assemblate precedentemente, risolvere numericamente l'equazione di diffusione-trasporto per $\alpha=0.01$, $\beta=1$. Confrontare la soluzione ottenuta con quella esatta, sapendo l'espressione di quest'ultima è

$$u(x)=\frac{e^{(x\alpha)/\beta}-1}{e^{\alpha/\beta}-1}$$

Si commenti il risultato.

In [ ]:
# dati del problema
alpha = 0.01
beta = 1

# assemblaggio matrice di sistema
...

# gestione condizioni di Dirichlet
dirichlet_nodes = [...]
dirichlet_values = [...]

bc_values = np.zeros(Nele + 1)
bc_values[dirichlet_nodes] = dirichlet_values

# costruzione matrice di restrizione R
keep_dof = np.ones(Nele + 1, dtype=bool)
keep_dof[dirichlet_nodes] = False

R = ...

# Restrizione di A e del termine noto
A_0 = ...
rhs_0 = ...

# risoluzione sistema lineare
u_0 = ...

# aggiunta condizione di Dirichlet
uh = ...

In [ ]:
# soluzione esatta
uex = lambda x: (np.exp(x * beta / alpha) - 1) / (np.exp(beta / alpha) - 1)

# soluzione numerica interpolata
interp_uh = ...

# x per il plot
xplot = np.linspace(0, 1, 1000)

# rappresentazione grafica
plt.figure(figsize=(5, 4))
plt.plot(xplot, interp_uh(xplot), "-", label="Soluzione FEM")
plt.plot(xplot, uex(xplot), "--", label="Soluzione esatta")
plt.legend()
plt.show()

**Commento**: siamo in un regime di trasporto dominante, in cui il coefficiente $\beta$, che rappresenta la velocità di trasporto, è molto maggiore del coefficiente diffusivo $\alpha$. In questo caso, dal punto di vista numerico, si osservano instabilità nella soluzione approssimata. In particolare, si osserva che la soluzione è costante nella maggior parte del dominio, mentre subisce una rapida variazione in corrispondenza dell'estremo destro. Questa regione in cui la soluzione varia è detta strato limite (boundary layer) ed è responsabile delle maggiori oscillazioni numeriche. La scelta del passo $h$ della griglia non è sufficientemente piccola per rappresentare lo strato limite. Esistono diversi metodi per risolvere o mitigare questo problema. Uno di questi consiste nel ridurre il passo di discretizzazione $h$, migliorando così la risoluzione numerica della soluzione. In particolare, si definisce il numero di Peclet come segue:

\begin{equation}
Pe = \frac{\beta h}{2\alpha}.
\end{equation}

Per evitare instabilità numeriche, occorre scegliere $h$ in modo tale che $Pe<1$. Con i dati del problema: $h<0.02$.

<mark>**Esercizio 2.4**</mark></br>

Ripetere l'Esercizio 2.3, variando il valore di $h$, in modo da ottenere una soluzione numerica stabile. 

# Equazioni di diffusione-trasporto-reazione

Si consideri il seguente problema differenziale, descrivente un fenomeno di diffusione-trasporto-reazione (stazionario)

$$
\begin{cases}
(-\alpha u'+\beta u)'+\gamma u = f, & x\in(0,1)\\
u(0)=u(1)=0.
\end{cases}
$$

dove $\alpha>0,\beta\neq0,\gamma>0$ sono opportuni coefficienti. La corrispondente formulazione debole è
</br></br>
$$\alpha\int_0^1u'v'dx-\beta\int_0^1uv'dx+\gamma\int_0^1uvdx=\int_0^1fvdx.$$

<mark>**Esercizio 3.1**</mark></br>

Sia $V_h$ lo spazio elementi finiti di grado $r=1$ e passo $h=0.01$. Assemblare le matrici associate alle forme bilineari, senza imporre alcuna condizione al bordo.


In [ ]:
from fem_utils import Grid, fun2dof, create_restriction, diffusion, transport, mass

# estremi del dominio
a = 0
b = 1
# dimensione elementi
h = 0.01
# numero elementi
Nele = int((b - a) / h)

# creazione griglia
...

# assemblaggio matrici di diffusione, trasporto e reazione
...

<mark>**Esercizio 3.2**</mark></br>

Sfruttando le matrici appena assemblate, risolvere numericamente l'equazione di diffusione-trasporto-reazione per $\alpha=1$, $\beta=2$ e $\gamma=3$. Si ponga $f\equiv-1$. Confrontare la soluzione ottenuta con quella esatta, sapendo l'espressione di quest'ultima è:

$$u(x)=C_1 e^{-x}+C_2e^{3x}-\frac{1}{3},$$

dove $C_2=\frac{1}{3}\frac{e-1}{e^4-1}$ e $C_1=\frac{1}{3}-C_2$.

La matrice di rigidezza totale è definita come $A = A_{\mathrm{diff}} + A_{\mathrm{trasp}} + A_{\mathrm{reaz}}$, includendo i coefficienti di diffusione, reazione e trasporto.

In [ ]:
# assemblaggio termine noto
...

# assemblaggio matrice di rigidezza
alpha, beta, gamma = 1, 2, 3
A = ...

# gestione condizioni di Dirichlet
dirichlet_nodes = [...]
dirichlet_values = [...]

bc_values = np.zeros(Nele + 1)
bc_values[dirichlet_nodes] = dirichlet_values
keep_dof = np.ones(Nele + 1, dtype=bool)
keep_dof[dirichlet_nodes] = False

R = ...
print(R)

A_0 = ...
rhs_0 = ...

# risoluzione sistema lineare
u_0 = ...
# aggiunta condizione di Dirichlet
u = ...

In [12]:
e = np.exp(1)
c2 = (e - 1.0) / (e**4 - 1.0) / 3.0
c1 = 1.0 / 3.0 - c2
uex = lambda x: c1 * np.exp(-x) + c2 * np.exp(3 * x) - 1.0 / 3.0

In [ ]:
xplot = np.linspace(0, 1, 1000)

plt.figure(figsize=(4, 3))
plt.plot(grid.nodes, u, "-", label="Soluzione FEM")
plt.plot(xplot, uex(xplot), "--", label="Soluzione esatta")
plt.legend()
plt.show()

<mark>**Esercizio 3.3**</mark></br>

Provate a ripetere l'Es. 3.2 per diversi valori di $b \ge 0$. Come cambia la soluzione numerica?

In [ ]:
# tutte le soluzioni numeriche nello stesso grafico al variare di beta
plt.figure(figsize=(6, 4))

for beta in ...
    A = ...

    # risoluzione sistema lineare
    ...

    plt.plot(grid.nodes, u, "-", label=f"$\\beta$ = {beta}")

plt.title("Soluzioni FEM per diversi valori di $\\beta$")
plt.xlabel("x")
plt.ylabel("u(x)")
plt.legend(loc="center left", bbox_to_anchor=(1.05, 0.5))
plt.grid(True, alpha=0.3)
plt.show()


Nel caso $\beta=0$, l'equazione va incontro esclusivamente a diffusione e reazione. La presenza del trasporto, invece, produce uno spostamento della soluzione verso destra, rendendola asimmetrica. Al crescere del trasporto, si tende ad un problema a trasporto dominante, con la creazione di uno strato limite per $x \rightarrow 1$.

**Commento**: combinando tutti gli strumenti visti nei Laboratori 10 e 11 è possibile risolvere problemi di diffusione-reazione-trasporto con condizioni di Dirichlet o Neumann. 